Dirichlet Lame Indirect Method
=============================

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.bem import *
from ngsolve import Projector, Preconditioner
from ngsolve.krylovspace import CG

Define the geometry $\Omega \subset \mathbb R^3$ and create a mesh:

In [ ]:
sp = Glue ( Sphere( (0,0,0), 1).faces)
mesh = Mesh( OCCGeometry(sp).GenerateMesh(maxh=0.5)).Curve(3)

$$V^{\text{Lame}} \, j \;=\; (\tfrac{1}{2}I + K^{\text{Lame}}) u_0$$
with
$$K^{\text{Lame}}u = Ku - V (M(u)) + 2\mu V^{\text{Lame}} (M(u))$$

In [ ]:
fesL2 = VectorValued(SurfaceL2(mesh, order=3, dual_mapping=False))
fesH1 = VectorH1(mesh, order=4)

print (fesL2.ndof)
print (fesH1.ndof)

u,v = fesL2.TnT()
uH1, vH1 = fesH1.TnT()

In [ ]:
n = specialcf.normal(3)
def MM(u) : 
    g = Grad(u).Trace()
    return g.trans * n - Trace(g) * n

In [ ]:
p0 = CF( (2,2,2) )
X = CF( (x,y,z) )

E, nu = 210, 0.2
mu = E/(2*(1+nu))
alpha = (1+nu)/((1-nu)*2*E)
norm = Norm(X-p0)
lapkernel = alpha/(4*pi) * 1/norm

u0 = (3-4*nu) * lapkernel * CF( (1,0,0) ) \
   + lapkernel/norm**2 * (X-p0) * (X-p0)[0] 

u0 *= 1000
# rhs = LinearForm (u0*v.Trace()*ds(bonus_intorder=3)).Assemble()
u_0 = GridFunction(fesH1)
u_0.Set(u0, definedon=mesh.Boundaries(".*"))
Draw (u_0, mesh)

In [ ]:
j = GridFunction(fesL2)
pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()

with TaskManager():
    V = LameSL(u*ds,E,nu) *v*ds
    M = BilinearForm( uH1 * v * ds(bonus_intorder=3)).Assemble()
    K = (LaplaceDL(uH1 * ds)*v*ds).mat - (LaplaceSL(MM(uH1)*ds) * v *ds).mat + 2*mu*(LameSL(MM(uH1)*ds, E, nu) * v *ds).mat    
    rhs = ( (0.5 * M.mat + K)*u_0.vec).Evaluate()
    CG(mat = V.mat, pre=pre, rhs = rhs, sol=j.vec, tol=1e-8, maxsteps=100, initialize=False, printrates=True)

In [ ]:
Draw (j, order=3);

In [ ]:
lam = E * nu / ((1+nu) * (1-2*nu))
grad_u = CF( (u0.Diff(x), u0.Diff(y), u0.Diff(z)) )
eps_u = 1/2 * (grad_u.Reshape((3,3)) + grad_u.Reshape((3,3)).trans) 
sigma_1 = lam * (u0.Diff(x)[0] + u0.Diff(y)[1] + u0.Diff(z)[2])
sigma_1 = CF((sigma_1, 0, 0, 0, sigma_1, 0, 0, 0, sigma_1)).Reshape((3,3))
sigma = sigma_1 + 2 * mu * eps_u
uexa = sigma * n
Draw (uexa, mesh, draw_vol=False, order=3);
Draw( j - uexa, mesh, draw_vol=False, order=3)

In [ ]:
vismesh = (WorkPlane().RectangleC(4,4).Face()*Sphere((0,0,0),1)).GenerateMesh(maxh=0.1).Curve(4)
sol = GridFunction(VectorH1(vismesh,order=3))

SL = (LameSL(u*ds(bonus_intorder=4), E, nu)*v*ds).GetPotential(j)
# DL = (LaplaceDL(uH1 * ds)*v*ds).GetPotential(u_0) - (LaplaceSL(MM(uH1)*ds)*v*ds).GetPotential(u_0) + 2 * mu * (LameSL(MM(uH1)*ds, E, nu)*v*ds).GetPotential(u_0)    
DL = LaplaceDL(uH1 * ds)(u_0) - LaplaceSL(MM(uH1)*ds)(u_0) + 2 * mu * LameSL(MM(uH1)*ds, E, nu)(u_0)    
repformula = SL-DL 
sol.Set (repformula, definedon=vismesh.Boundaries(".*"))
Draw (sol, vismesh, order=3);
Draw (u0, vismesh, order=3);
Draw (sol-u0, vismesh, order=3);